In [1]:
import xarray as xr
import geopandas as gpd
import rioxarray  # noqa: F401 — extends xarray with .rio accessor
 
# ── Configuration ────────────────────────────────────────────────────────────
NC_FILE   = "/Users/miguelsilveira/Documents/GitHub/marineheatwaves/oisst_daily/sst_merged.nc"          # Path to your NetCDF file
SHP_FILE  = "/Users/miguelsilveira/Documents/GitHub/marineheatwaves/200m/200m.shp"      # Path to your shapefile
OUTPUT    = "output_clipped.nc" # Output file path
 
# Coordinate dimension names in your NetCDF (adjust if different)
X_DIM = "lon"   # or "longitude", "x"
Y_DIM = "lat"   # or "latitude",  "y"
 
# Set to True to keep only grid cells whose CENTER falls inside the shape.
# Set to False to keep any cell that touches the shape boundary (default).
ALL_TOUCHED = False
# ─────────────────────────────────────────────────────────────────────────────
 
 
def clip_nc_by_shapefile(nc_file, shp_file, output, x_dim, y_dim, all_touched):
    print(f"Loading NetCDF: {nc_file}")
    ds = xr.open_dataset(nc_file)
    print(ds)
 
    # Assign CRS and spatial dims
    ds = ds.rio.set_spatial_dims(x_dim=x_dim, y_dim=y_dim, inplace=True)
    ds = ds.rio.write_crs("EPSG:4326", inplace=True)
 
    print(f"\nLoading shapefile: {shp_file}")
    shp = gpd.read_file(shp_file)
    print(f"  CRS: {shp.crs}  |  Features: {len(shp)}")
 
    # Reproject shapefile to match NetCDF CRS if needed
    if shp.crs.to_epsg() != 4326:
        print("  Reprojecting shapefile to EPSG:4326 ...")
        shp = shp.to_crs("EPSG:4326")
 
    print("\nClipping ...")
    clipped = ds.rio.clip(
        shp.geometry,
        shp.crs,
        drop=True,           # drop rows/cols entirely outside the shape
        invert=False,        # set True to mask INSIDE and keep OUTSIDE
        all_touched=all_touched,
    )
 
    print(f"\nSaving to: {output}")
    clipped.to_netcdf(output)
    print("Done!")
    return clipped
 
 
if __name__ == "__main__":
    clip_nc_by_shapefile(NC_FILE, SHP_FILE, OUTPUT, X_DIM, Y_DIM, ALL_TOUCHED)

Loading NetCDF: /Users/miguelsilveira/Documents/GitHub/marineheatwaves/oisst_daily/sst_merged.nc
<xarray.Dataset> Size: 67GB
Dimensions:  (time: 16190, lat: 720, lon: 1440)
Coordinates:
  * time     (time) datetime64[ns] 130kB 1981-09-01 1981-09-02 ... 2025-12-28
  * lat      (lat) float32 3kB -89.88 -89.62 -89.38 -89.12 ... 89.38 89.62 89.88
  * lon      (lon) float32 6kB 0.125 0.375 0.625 0.875 ... 359.4 359.6 359.9
Data variables:
    sst      (time, lat, lon) float32 67GB ...
Attributes:
    Conventions:    CF-1.5
    title:          NOAA High-resolution Blended Analysis: Daily Values using...
    institution:    NOAA/NCDC
    source:         NOAA/NCDC  ftp://eclipse.ncdc.noaa.gov/pub/OI-daily-v2/
    history:        Tue Mar  3 17:57:33 2026: ncrcat sst.day.mean.1981.nc sst...
    dataset_title:  NOAA Daily Optimum Interpolation Sea Surface Temperature
    References:     https://www.psl.noaa.gov/data/gridded/data.noaa.oisst.v2....
    comment:        Reynolds, et al.(2007) Daily H

: 